## 1 · Imports, device, dataset

In [ ]:
from datasets import load_dataset

from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import math
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")


## 2 · Hyperparameters & DataLoader

In [ ]:
batch_size = 64
embedding_dim = 512
num_heads = 8
ff_hidden_dim = 2048
num_layers = 6
max_length = 256

train_loader = DataLoader(
    ds["train"],
    batch_size=batch_size,
    shuffle=True
)


## 3 · Write SentencePiece training corpus

In [ ]:
with open("corpus.txt", "w", encoding="utf-8") as f:

    for item in ds["train"]:

        en = item["translation"]["en"]
        ja = item["translation"]["ja"]

        f.write(en + "\n")
        f.write(ja + "\n")


## 4 · Train SentencePiece model

In [ ]:
spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="translator_sp",
    vocab_size=16000,
    model_type="Unigram",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)


## 5 · Load SentencePiece model & verify special IDs

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("translator_sp.model")

# FIX: verify that pad_id is 0 as configured.
# If sp.pad_id() returns -1 the model was loaded without the pad_id spec
# and pad_sequence will fill tensors with -1, crashing nn.Embedding.
assert sp.pad_id() == 0, (
    f"sp.pad_id() returned {sp.pad_id()}.  "
    "The SentencePiece model must be trained with pad_id=0. "
    "Re-run the SentencePieceTrainer cell and reload."
)
print(f"PAD={sp.pad_id()}  BOS={sp.bos_id()}  EOS={sp.eos_id()}  VOCAB={sp.get_piece_size()}")


## 6 · Tokenize helper (reference – not used by training loop)

In [ ]:
def tokenize(example):

    source = (
        [sp.bos_id()]
        + sp.encode(
            example["translation"]["en"],
            out_type=int
        )
        + [sp.eos_id()]
    )

    target = (
        [sp.bos_id()]
        + sp.encode(
            example["translation"]["ja"],
            out_type=int
        )
        + [sp.eos_id()]
    )

    return {
        "source": source,
        "target": target
    }


## 7 · Vocabulary constants

In [ ]:
VOCAB_SIZE = sp.get_piece_size()
PAD_ID     = sp.pad_id()
BOS_ID     = sp.bos_id()
EOS_ID     = sp.eos_id()

print(f"VOCAB_SIZE={VOCAB_SIZE}  PAD_ID={PAD_ID}  BOS_ID={BOS_ID}  EOS_ID={EOS_ID}")


## 8 · MultiHeadAttention

> **FIX (Bug 5):** mask convention changed to boolean `True = CAN attend`.
> `masked_fill(~mask, ...)` replaces `masked_fill(mask == 0, ...)` so that
> padding masks and causal masks share the same polarity.

In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads
    ):
        super().__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.query = nn.Linear(embedding_dim, embedding_dim)
        self.key   = nn.Linear(embedding_dim, embedding_dim)
        self.value = nn.Linear(embedding_dim, embedding_dim)
        self.fc_out = nn.Linear(embedding_dim, embedding_dim)

    def forward(
        self,
        query,
        key,
        value,
        mask=None   # bool tensor, True = position is ALLOWED to be attended to
    ):

        batch_size = query.shape[0]

        Q = self.query(query)
        K = self.key(key)
        V = self.value(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            # FIX: use ~mask so True=attend, False=block (consistent with both
            #      causal masks and padding masks built elsewhere in this file).
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, -1, self.embedding_dim)
        output = self.fc_out(output)

        return output


## 9 · EncoderBlock

In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim):
        super().__init__()

        self.attention = MultiHeadAttention(embedding_dim, num_heads)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, ff_hidden_dim),
            nn.GELU(),
            nn.Linear(ff_hidden_dim, embedding_dim)
        )

    def forward(self, x, mask=None):

        attn = self.attention(x, x, x, mask)
        x = self.norm1(x + attn)

        ffn = self.ffn(x)
        x = self.norm2(x + ffn)

        return x


## 10 · DecoderBlock

> **FIX (Bug 4):** `cross_attention` now receives `src_mask` so the decoder
> does not attend to padding positions in the encoder output.
> Previously the mask argument was missing entirely.

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim):
        super().__init__()

        self.self_attention  = MultiHeadAttention(embedding_dim, num_heads)
        self.cross_attention = MultiHeadAttention(embedding_dim, num_heads)

        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.norm3 = nn.LayerNorm(embedding_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, ff_hidden_dim),
            nn.GELU(),
            nn.Linear(ff_hidden_dim, embedding_dim)
        )

    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):

        # Masked self-attention (causal)
        attn = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + attn)

        # FIX: pass src_mask so decoder never attends to encoder padding tokens
        cross = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + cross)

        ffn = self.ffn(x)
        x = self.norm3(x + ffn)

        return x


## 11 · Encoder

> **FIX (Bug 3):** `forward` now accepts and forwards `mask` to every
> `EncoderBlock` so padding tokens are blocked from self-attention.

In [ ]:
class Encoder(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim, num_layers):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderBlock(embedding_dim, num_heads, ff_hidden_dim)
            for _ in range(num_layers)
        ])

    # FIX: accept mask and pass it to every EncoderBlock
    def forward(self, x, mask=None):

        for layer in self.layers:
            x = layer(x, mask)

        return x


## 12 · Decoder

> **FIX:** `forward` forwards both `tgt_mask` and `src_mask` to every
> `DecoderBlock`.

In [ ]:
class Decoder(nn.Module):

    def __init__(self, embedding_dim, num_heads, ff_hidden_dim, num_layers):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderBlock(embedding_dim, num_heads, ff_hidden_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, tgt_mask, src_mask=None):

        for layer in self.layers:
            x = layer(x, encoder_output, tgt_mask, src_mask)

        return x


## 13 · causal_mask helper

> **FIX (Bug 5):** returns a `bool` tensor (`True` = can attend) and accepts
> a `device` argument so the mask lands on the right device in one call.

In [ ]:
def causal_mask(seq_len, device):
    # True  = position is allowed to be attended to
    # False = position is blocked (future / upper-triangle)
    mask = torch.tril(
        torch.ones(seq_len, seq_len, device=device, dtype=torch.bool)
    )
    return mask.unsqueeze(0).unsqueeze(0)   # [1, 1, T, T]


## 14 · Transformer

> **FIX (Bug 1 — primary crash):** `forward` now clamps `src_pos` and `tgt_pos`
> to `[0, max_length-1]` so a sequence longer than `max_length` does **not**
> generate out-of-range indices into `position_embedding`.
>
> **FIX (Bug 3 & 4):** builds a `src_key_padding_mask` and passes it through
> the encoder and into decoder cross-attention.

In [ ]:
class Transformer(nn.Module):

    def __init__(
        self,
        VOCAB_SIZE,
        embedding_dim,
        num_heads,
        ff_hidden_dim,
        num_layers,
        max_length
    ):
        super().__init__()

        self.max_length = max_length

        self.token_embedding    = nn.Embedding(VOCAB_SIZE, embedding_dim)
        self.position_embedding = nn.Embedding(max_length, embedding_dim)

        self.encoder = Encoder(embedding_dim, num_heads, ff_hidden_dim, num_layers)
        self.decoder = Decoder(embedding_dim, num_heads, ff_hidden_dim, num_layers)

        self.fc_out = nn.Linear(embedding_dim, VOCAB_SIZE)

    def forward(self, source_input, target_input, pad_id=0):

        B, src_len = source_input.shape
        _, tgt_len = target_input.shape

        # ── FIX (Bug 1): clamp position indices to [0, max_length-1] ──────────
        # Without this, any sequence longer than max_length produces indices
        # that are out-of-range for position_embedding → CUDA device-side assert.
        src_pos = torch.arange(src_len, device=source_input.device).unsqueeze(0)
        src_pos = src_pos.clamp(max=self.max_length - 1)

        tgt_pos = torch.arange(tgt_len, device=target_input.device).unsqueeze(0)
        tgt_pos = tgt_pos.clamp(max=self.max_length - 1)

        src = self.token_embedding(source_input) + self.position_embedding(src_pos)
        tgt = self.token_embedding(target_input) + self.position_embedding(tgt_pos)

        # ── FIX (Bug 3): build src padding mask and pass it to the encoder ─────
        # Shape: [B, 1, 1, src_len] — broadcasts over (heads, query_positions)
        # True  = real token (attend)   False = padding (block)
        src_key_padding_mask = (source_input != pad_id)            # [B, src_len]
        src_mask = src_key_padding_mask.unsqueeze(1).unsqueeze(2)  # [B, 1, 1, src_len]

        encoder_output = self.encoder(src, mask=src_mask)

        # Causal mask for decoder self-attention
        tgt_mask = causal_mask(tgt_len, device=target_input.device)  # [1,1,T,T]

        # ── FIX (Bug 4): pass src_mask to decoder so cross-attn ignores padding ─
        decoder_output = self.decoder(tgt, encoder_output, tgt_mask, src_mask)

        logits = self.fc_out(decoder_output)

        return logits


## 15 · Model, criterion, optimizer

In [ ]:
model = Transformer(
    VOCAB_SIZE=VOCAB_SIZE,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_hidden_dim=ff_hidden_dim,
    num_layers=num_layers,
    max_length=max_length
).to(device)

pad_id = sp.pad_id()

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)


## 16 · AMP / GradScaler setup

In [ ]:
import os
from torch.amp import autocast, GradScaler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

scaler = GradScaler("cuda")


## 17 · Single-batch sanity check

> **FIX (Bug 1):** checks `source_input.shape[1] <= max_length` — the check
> that was missing before.  The old checks only verified token *values*, not
> sequence *length*.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

batch = next(iter(train_loader))

source_texts = batch["translation"]["en"]

print(type(source_texts))
print(len(source_texts))

tokens = [
    torch.tensor(
        ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
        dtype=torch.long
    )
    for text in source_texts
]

print("tokenization success")

padded = pad_sequence(
    tokens,
    batch_first=True,
    padding_value=sp.pad_id()
)

print("padding success")
print("shape       :", padded.shape)
print("max token   :", padded.max().item())
print("min token   :", padded.min().item())
print("seq_len     :", padded.shape[1], "(must be <=", max_length, ")")

# FIX: check sequence length, NOT just token values
assert padded.shape[1] <= max_length, (
    f"seq_len={padded.shape[1]} exceeds max_length={max_length} "
    "even after truncation – check the slicing logic above."
)
print("Sequence length check passed ✓")


## 18 · Token embedding smoke test

In [ ]:
source_input = padded.to(device)

print("Moved to CUDA")

with torch.no_grad():
    emb = model.token_embedding(source_input)

print("Embedding success")
print(emb.shape)


## 19 · Position embedding smoke test

In [ ]:
source_input = padded.to(device)
src_len = source_input.shape[1]

src_pos = torch.arange(src_len, device=source_input.device).unsqueeze(0)

print("Position indices range: 0 –", src_pos.max().item(), "(max_length-1 =", max_length - 1, ")")
assert src_pos.max().item() < max_length, (
    f"src_pos max={src_pos.max().item()} >= max_length={max_length}"
)

with torch.no_grad():
    token_emb = model.token_embedding(source_input)

print("Token embedding success")

with torch.no_grad():
    pos_emb = model.position_embedding(src_pos)

print("Position embedding success")

src = token_emb + pos_emb
print("Combined embedding success, shape:", src.shape)


## 20 · Encoder smoke test

In [ ]:
with torch.no_grad():
    enc_out = model.encoder(src)

print("Encoder success")
print(enc_out.shape)
print("embedding_dim:", embedding_dim)
print("num_heads    :", num_heads)


## 21 · Training loop

Fixes applied here:

- **Bug 1 (primary crash):** sequences truncated to `max_length` tokens *before* padding.
- **Bug 1 cont.:** explicit `assert shape[1] <= max_length` added after padding.
- **Bug 2:** `target_output` range-checked before it reaches `CrossEntropyLoss`.
- **Cleanup:** removed per-batch `print` spam (kept tqdm progress bar).
- **Clarity:** `pad_id` passed explicitly to `model.forward`.

In [ ]:
from tqdm import tqdm
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence

print("SentencePiece vocab :", sp.get_piece_size())
print("Model vocab         :", model.token_embedding.num_embeddings)

assert sp.get_piece_size() == model.token_embedding.num_embeddings, \
    "SentencePiece vocab and model vocab do not match!"

epochs = 6
scaler = GradScaler("cuda")

for epoch in range(epochs):

    model.train()
    total_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs}"
    )

    for batch_idx, batch in enumerate(progress_bar):

        source_texts = batch["translation"]["en"]
        target_texts = batch["translation"]["ja"]

        # ── FIX (Bug 1): truncate every sequence to max_length BEFORE padding ──
        # Without truncation, pad_sequence pads to the longest sequence in the
        # batch. If that length > max_length, position indices exceed the size
        # of position_embedding → CUDA device-side assert in nn.Embedding.
        source_tokens = [
            torch.tensor(
                ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
                dtype=torch.long
            )
            for text in source_texts
        ]

        target_tokens = [
            torch.tensor(
                ([sp.bos_id()] + sp.encode(text, out_type=int) + [sp.eos_id()])[:max_length],
                dtype=torch.long
            )
            for text in target_texts
        ]

        source_input = pad_sequence(
            source_tokens,
            batch_first=True,
            padding_value=sp.pad_id()
        ).to(device)

        target_full = pad_sequence(
            target_tokens,
            batch_first=True,
            padding_value=sp.pad_id()
        ).to(device)

        target_input  = target_full[:, :-1]
        target_output = target_full[:, 1:]

        # ── FIX (Bug 1 cont.): assert sequence length, not just token values ───
        # The OLD checks (source_input.max() < VOCAB_SIZE) verified token IDs
        # (0‒15999). They said nothing about sequence length. A 300-token
        # sequence has a perfectly valid max token ID but crashes position_embedding.
        assert source_input.shape[1] <= max_length, (
            f"src_len={source_input.shape[1]} > max_length={max_length}"
        )
        assert target_input.shape[1] <= max_length, (
            f"tgt_len={target_input.shape[1]} > max_length={max_length}"
        )

        # Token value sanity checks (kept from original)
        assert source_input.min() >= 0, \
            f"Negative source token: {source_input.min().item()}"
        assert target_input.min() >= 0, \
            f"Negative target token: {target_input.min().item()}"
        assert source_input.max() < VOCAB_SIZE, \
            f"SRC token {source_input.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"
        assert target_input.max() < VOCAB_SIZE, \
            f"TGT token {target_input.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"

        # ── FIX (Bug 2): also check target_output before it hits the loss ──────
        # The original code only checked target_INPUT. target_OUTPUT is what
        # CrossEntropyLoss actually receives; invalid values there crash on GPU.
        assert target_output.min() >= 0, \
            f"Negative value in target_output: {target_output.min().item()}"
        assert target_output.max() < VOCAB_SIZE, \
            f"target_output token {target_output.max().item()} >= VOCAB_SIZE {VOCAB_SIZE}"

        optimizer.zero_grad()

        with autocast("cuda"):

            predictions = model(
                source_input,
                target_input,
                pad_id=PAD_ID
            )

            loss = criterion(
                predictions.reshape(-1, VOCAB_SIZE),
                target_output.reshape(-1)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        avg_loss = total_loss / (batch_idx + 1)
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{avg_loss:.4f}"
        )

    avg_epoch_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1}/{epochs} Completed | Average Loss: {avg_epoch_loss:.4f}\n")

    torch.save(model.state_dict(), f"translator_epoch_{epoch+1}.pth")


## 22 · Post-training target_output check

In [ ]:
print("VOCAB_SIZE:", VOCAB_SIZE)
print("Max target:", target_output.max().item())
print("Min target:", target_output.min().item())


## 23 · Memory cleanup

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()


## 24 · Inference: `translate()`

> **FIX (Bug 1):** source tokens truncated to `max_length` so inference
> cannot exceed the position embedding table either.

In [ ]:
def translate(
    text,
    model,
    sp,
    device,
    max_length=256
):

    model.eval()

    with torch.no_grad():

        # FIX: truncate source to max_length (same rule as training)
        source_tokens = (
            [sp.bos_id()]
            + sp.encode(text, out_type=int)
            + [sp.eos_id()]
        )[:max_length]

        source_input = torch.tensor(
            [source_tokens],
            dtype=torch.long,
            device=device
        )

        generated = torch.tensor(
            [[sp.bos_id()]],
            dtype=torch.long,
            device=device
        )

        for _ in range(max_length):

            output = model(source_input, generated, pad_id=sp.pad_id())

            next_token = output[:, -1, :].argmax(dim=-1)

            generated = torch.cat(
                [generated, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == sp.eos_id():
                break

        ids = generated[0].tolist()
        ids = [
            x for x in ids
            if x not in [sp.pad_id(), sp.bos_id(), sp.eos_id()]
        ]

        return sp.decode(ids)


## 25 · Run a translation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

english_text = "Hello"

translation = translate(
    text=english_text,
    model=model,
    sp=sp,
    device=device
)

print("\nTranslation:")
print(translation)
